
> # **Structural_module for Protease Inhibitor Screening**




**Model type**: One-class unsupervised Autoencoder (PyOD, PyTorch) |
Task: Structural filtering of protease inhibitor–like protein structures

***What it does***

Learns the structural embedding manifold of known protease inhibitors (PIs) and assigns a reconstruction error to each input.

Low error: PI-like structure | High error: Structurally inconsistent / non-PI-like

**Data**

Training set: 17,889 curated PI structures |
Source: MEROPS → UniProt mapping → AlphaFold DB |
Embeddings: RCSB protein structure embeddings
https://github.com/rcsb/rcsb-embedding-model

**Input**

Format: Fixed-length RCSB embeddings |
Required: Standardization using provided scaler.pkl

**Architecture & Training**

Fully connected encoder–decoder with bottleneck |
Batch normalization + dropout |
Adam optimizer, MSE loss |
Hyperparameters tuned via Optuna (TPE)

**Intended Use**

Large-scale structural pre-filtering of PI candidates |
Quality control for predicted or generated protein structures

**Limitations**

Not intended for functional annotation or clinical use

# **Colab Runtime Instructions**

Runtime → Change runtime type → Hardware accelerator: GPU → GPU type: T4 → Save → Run all

Upload the protein structure file as .cif format when prompted and wait a few minutes for results. If you have multiple strcutures compress it as .rar and upload.

In [ ]:
# @title Install Dependencies
print("Installing dependencies...")
!pip install -q "git+https://github.com/rcsb/rcsb-embedding-model.git" > /dev/null
!pip install -q esm > /dev/null
!pip install -q biopython > /dev/null
print("Dependencies installed.")

Installing dependencies...
Dependencies installed.


In [ ]:
# @title Upload Structures (RAR/ZIP supported) & Generate Embeddings
import os
import csv
import zipfile
import shutil
import subprocess
import numpy as np
import torch
from google.colab import files
from rcsb_embedding_model import RcsbStructureEmbedding
from tqdm.notebook import tqdm
output_csv = "RCSB_structures_embedding.csv"
chain_id = "A"  # @param {type:"string"}

subprocess.run(["apt-get", "install", "unrar"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

processing_folder = "/content/processing_input"
if os.path.exists(processing_folder):
    shutil.rmtree(processing_folder)
os.makedirs(processing_folder, exist_ok=True)

print("Upload your protein structure files (.rar or .zip):")
uploaded = files.upload()

if not uploaded:
    print("No files uploaded.")
else:
    model = RcsbStructureEmbedding()
    for filename, content in uploaded.items():
        file_path = os.path.join(processing_folder, filename)
        with open(file_path, 'wb') as f:
            f.write(content)

        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                zip_ref.extractall(processing_folder)
            os.remove(file_path)
        elif filename.lower().endswith(".rar"):
            subprocess.run(["unrar", "x", "-o+", "-inul", file_path, processing_folder])
            os.remove(file_path)
    structure_files = []
    for root, dirs, files_in_dir in os.walk(processing_folder):
        for filename in files_in_dir:
            if filename.lower().endswith((".pdb", ".cif")):
                structure_files.append(os.path.join(root, filename))

    header_written = False
    success_count = 0
    error_count = 0

    with open(output_csv, "w", newline="") as f:
        writer = csv.writer(f)
        for file_path in tqdm(structure_files, desc="Generating Embeddings"):
            filename = os.path.basename(file_path)
            protein_id = os.path.splitext(filename)[0]

            try:
                res_emb = model.residue_embedding(src_structure=file_path, chain_id=chain_id)
                struct_emb = model.aggregator_embedding(res_emb)

                if isinstance(struct_emb, torch.Tensor):
                    struct_emb_np = struct_emb.detach().cpu().numpy().flatten()
                else:
                    struct_emb_np = np.array(struct_emb).flatten()

                if not header_written:
                    header = ["ProteinID"] + [f"emb_{i}" for i in range(len(struct_emb_np))]
                    writer.writerow(header)
                    header_written = True

                writer.writerow([protein_id] + struct_emb_np.tolist())
                success_count += 1

            except Exception:
                error_count += 1

    print(f"\nDone. Processed: {success_count} | Errors: {error_count}")
    if success_count > 0:
        files.download(output_csv)

Upload your protein structure files (.rar or .zip):


Saving LLM_hits.rar to LLM_hits.rar


Generating Embeddings:   0%|          | 0/218 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

rcsb-embedding-model.pt:   0%|          | 0.00/690M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/esm/models/vqvae.py:283: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):  # type: ignore
/usr/local/lib/python3.12/dist-packages/esm/models/vqvae.py:172: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:345.)
  return data[ranges]



Done. Processed: 218 | Errors: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# @title Install dependencies for autoencoder model
!pip install -q pyod huggingface_hub joblib pandas torch

In [ ]:
# @title Download pretrained autoencoder model from huggingface repository for inference
import argparse
import os
import warnings
import numpy as np
import pandas as pd
import torch
import joblib
from huggingface_hub import hf_hub_download

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def download_hf_assets(repo_id):
    print(f"Downloading assets from {repo_id}...")
    scaler_path = hf_hub_download(repo_id=repo_id, filename="scaler.pkl")
    model_path = hf_hub_download(repo_id=repo_id, filename="trained_autoencoder_model.pkl")
    return scaler_path, model_path

def run_inference(input_csv, scaler_path, model_path, output_csv):
    print(f"Using device: {DEVICE}")
    scaler = joblib.load(scaler_path)
    model = joblib.load(model_path)
    model.device = DEVICE
    model.model.to(DEVICE)
    df = pd.read_csv(input_csv)
    protein_ids = df.iloc[:, 0].values
    X = df.iloc[:, 1:].values.astype(np.float32)

    X_scaled = scaler.transform(X)

    anomaly_scores = model.decision_function(X_scaled)
    proba_outlier = model.predict_proba(X_scaled, method="linear")[:, 1]

    confidence = model.predict_confidence(X_scaled) if hasattr(model, "predict_confidence") else np.ones(len(X_scaled))
    with torch.no_grad():
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32).to(DEVICE)
        recon = model.model(X_tensor).cpu().numpy()

    recon_mse = np.mean((X_scaled - recon) ** 2, axis=1)

    results_df = pd.DataFrame({
        "protein_id": protein_ids,
        "anomaly_score": anomaly_scores,
        "proba_outlier": proba_outlier,
        "confidence": confidence,
        "reconstruction_mse": recon_mse
    }).sort_values("anomaly_score", ascending=False)

    results_df.to_csv(output_csv, index=False)
    print(f"Saved output: {output_csv}")

if __name__ == "__main__":
    HF_REPO = "MuthuS97/structuralmodule-protease_inhibitors"
    INPUT_FILE = "RCSB_structures_embedding.csv"

    s_path, m_path = download_hf_assets(HF_REPO)

    run_inference(INPUT_FILE, s_path, m_path, "AE_probability_final_results.csv")

Using device: cuda
Saved output: AE_probability_final_results.csv
